# 01 — Validate culture grain

## Goal

ARMD stores microbiology results in long form, so a single culture can appear on many source rows. Before building anything downstream, I wanted to lock down the culture grain and make sure `fct_culture` is really one row per culture order.

I check three things here: key uniqueness, consistency of order-level fields, and row-count reconciliation from raw to staging to the culture fact.

In [3]:
%sql
select
    count(*) as culture_rows,
    count(distinct culture_key) as distinct_culture_keys,
    sum(
        case
            when has_order_level_conflict then 1
            else 0
        end
    ) as culture_orders_with_conflicts
from workspace.micro_dev.fct_culture

,culture_rows,distinct_culture_keys,culture_orders_with_conflicts
0,751075,751075,0


### Grain check

The fact table has 751,075 rows and 751,075 distinct culture keys, with no order-level conflicts. That gives me a clean culture-order fact to use as the parent grain for the rest of the pipeline.

In [4]:
%sql
select
    source_row_count,
    count(*) as culture_orders
from workspace.micro_dev.fct_culture
group by source_row_count
order by source_row_count

,source_row_count,culture_orders
0,1,632612
1,2,456
2,3,703
3,4,3257
4,5,6289
5,6,2641
6,7,6447
7,8,7331
8,9,7613
9,10,2488


### How much source detail rolls up to a culture

`source_row_count` shows how many long-form microbiology rows feed each culture. The spread is expected: cultures with multiple organisms and susceptibility results contribute more source rows.

This is the main reason I do not use the 2.24 million source rows as a culture denominator.

In [5]:
%sql
select
    'raw_armd_cohort' as model_name,
    count(*) as row_count
from workspace.micro_raw.raw_armd_cohort

union all

select
    'stg_armd_cohort' as model_name,
    count(*) as row_count
from workspace.micro_dev.stg_armd_cohort

union all

select
    'fct_culture' as model_name,
    count(*) as row_count
from workspace.micro_dev.fct_culture

,model_name,row_count
0,raw_armd_cohort,2241050
1,stg_armd_cohort,2241050
2,fct_culture,751075


### Row-count reconciliation

Raw and staging both stay at 2,241,050 rows. The drop to 751,075 in `fct_culture` is intentional and comes from changing the grain, not from filtering records out.

### Takeaway

The culture grain is stable and unique, so downstream culture-organism and susceptibility models can key back to it without inflating counts.